In [1]:
# import subprocess 
# import time

# command_base = "~/coriolis-2.x/src/alliance-check-toolkit/bin/crlenv.py "
# command_yosys = "doit clean_flow b2v "
# command = command_base + command_yosys

# # Run the command and capture output
# t0 = time.time()
# process = subprocess.run(command, shell=True, capture_output=True, text=True)
# t1 = time.time()
# print(f"It took yosys = {t1 - t0} sec")

In [2]:
def format_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = seconds % 60
    
    parts = []
    if hours > 0:
        parts.append(f"{hours}h")
    if minutes > 0 or hours > 0:
        parts.append(f"{minutes}m")
    parts.append(f"{secs:.2f}s")
    
    return " ".join(parts)

In [3]:
# stdout_list = process.stdout.split()

# index_yosys_stat_start = process.stdout.find("9. Printing statistics.")
# index_yosys_stat_end = process.stdout.find("10. Executing BLIF backend.")
# yosys_stat =  process.stdout[index_yosys_stat_start : index_yosys_stat_end] 
# print(yosys_stat)

# with open("yosys_stat", "w") as log_file:
#     log_file.write(yosys_stat)


def run_yosys(verbose=True):
    import subprocess 
    import time
    
    command_base = "~/coriolis-2.x/src/alliance-check-toolkit/bin/crlenv.py"
    command_yosys = "doit clean_flow b2v"
    command = command_base + " " + command_yosys
    
    # Run the command and capture output
    t0 = time.time()
    process = subprocess.run(command, shell=True, capture_output=True, text=True)
    t1 = time.time()

    
    print(f"It took yosys = {format_time(t1 - t0)}")

    # Save the portion stats of the output to a file
    stdout_list = process.stdout.split()
    index_yosys_stat_start = process.stdout.find("9. Printing statistics.")
    index_yosys_stat_end = process.stdout.find("10. Executing BLIF backend.")
    yosys_stat =  process.stdout[index_yosys_stat_start : index_yosys_stat_end] 
    if verbose:
        print(yosys_stat)
    

In [4]:
def extract_effective_density(text):
    import re
    
    match = re.search(r"Effective density\s+\.*\s+([\d.]+)%", text)
    if match:
        return float(match.group(1))
    return None  # or raise an exception if preferred



def extract_wire_length(text):
    import re
    
    # Extract last 'Wire Length Completion Ratio' wire length
    wire_length_match = re.findall(r"Wire Length Completion Ratio\s+\.*\s+[\d.]+%\s+\[([\d]+)um", text)
    if wire_length_match:
        wire_length = int(wire_length_match[-1])  # Take the last match
    else:
        wire_length = None  # If not found
    
    return wire_length


def extract_area_dimensions(text):
    import re
    # This regex captures decimal or integer numbers before 'um'
    match = re.search(r"Whole place area: <Box \d+um \d+um ([\d.]+)um ([\d.]+)um>", text)
    if match:
        width = float(match.group(1))
        height = float(match.group(2))
        return width, height
    return None, None


In [5]:
def run_gds():
    """
    Run gds
    Returns the effective density
    Also, it saves the wire length, chips area, and effective density in the file
    core_stats
    """
    
    import subprocess 
    import time
    
    command_base = "~/coriolis-2.x/src/alliance-check-toolkit/bin/crlenv.py"
    command_yosys = "doit gds"
    command = command_base + " " + command_yosys
    
    # Run the command and capture output
    t0 = time.time()
    process = subprocess.run(command, shell=True, capture_output=True, text=True)
    t1 = time.time()
    
    
    
    density = extract_effective_density(process.stdout)
    wire_length = extract_wire_length(process.stdout)
    dimensions = extract_area_dimensions(process.stdout)
    
    # if  density == None:
    #     print("ERROR")
    #     print(process.stderr)
        
    print(f"GDS: took = {format_time(t1 - t0)} . Effective desnsity =  {density}, Wire length = {wire_length}")

    with open("output.stdout.log", "w") as f:
        f.write(process.stdout)

    with open("output.stderr.log", "w") as f:
        f.write(process.stderr)



    return density, wire_length, dimensions


In [6]:
def update_doDesign(scalar_a, scalar_b, connectors_margin):
    filename="doDesign.py"
    
    with open(filename, "r") as file:
        lines = file.readlines()
    
    new_lines = []
    for line in lines:
        if line.strip().startswith("scalar_a") and "scalar_b" in line:
            new_line = f"scalar_a, scalar_b = {scalar_a}, {scalar_b}  # updated values\n"
        elif line.strip().startswith("connectors_margin"):
            new_line = f"connectors_margin = {connectors_margin}  # updated value\n"
        else:
            new_line = line
        new_lines.append(new_line)
    
    with open(filename, "w") as file:
        file.writelines(new_lines)

In [7]:
!pwd
scalar_a, scalar_b, connectors_margin = 650//4 - 10, 100//4 - 5, 2
update_doDesign(scalar_a, scalar_b, connectors_margin)
run_yosys()
run_gds()


/home/fjjf/circuits/experiments/rot
It took yosys = 4.68s
9. Printing statistics.

=== rot ===

   Number of wires:                486
   Number of wire bits:           1751
   Number of public wires:          27
   Number of public wire bits:    1292
   Number of memories:               0
   Number of memory bits:            0
   Number of processes:              0
   Number of cells:                389
     inv_x0                          5
     mux2_x1                       384

Removed 0 unused cells and 149 unused wires.


GDS: took = 4.36s . Effective desnsity =  63.9, Wire length = None


(63.9, None, (121.6, 115.2))

In [8]:
# optimize_effective_area

with open("gds_stat.log", "a") as f:
    f.write("")

run_yosys()

scalar_a, scalar_b, connectors_margin = 650//2, 100//2, 2
print(f"Trying with a={scalar_a}, b={scalar_b}")

threshold_min, threshold_max = 70, 100

update_doDesign(scalar_a, scalar_b, connectors_margin)
density = 0

history_wire_lengths = []
history_density = []
history_dimensions = []
# reduce or increase the density
while density < 60 or  density >= 70 or density > 100:
    print(f"Going to try with a={scalar_a}, b={scalar_b}")
    update_doDesign(scalar_a, scalar_b, connectors_margin)
    density, wire_length, dimensions = run_gds()
    #print(f"Trying with a={scalar_a}, b={scalar_b}")
    # use multiplication here to get a better density
    if density < 25:# or density > 120:
        scalar_a = 2 * scalar_a / density 
        scalar_b = 2 * scalar_b / density
    
    elif density > 400:
        scalar_a = scalar_a * ( density / 50) 
        scalar_b = scalar_b * ( density / 50)       
        #print(f"factor = {(1/ (density / 100) )}, scalar_a = {scalar_a}, scalar_b = {scalar_b} ")
        
    # increase the chip by adding 10, and 5
    elif density > 50 and density < 59:
        scalar_a = scalar_a - 13
        scalar_b = scalar_b - 2
    
    elif density >= 70 and density < 400:
        scalar_a = scalar_a + 13
        scalar_b = scalar_b + 2


    # test only substract and add
    # increase the chip by adding 10, and 5
    if  density < 100:
        scalar_a = scalar_a - 13
        scalar_b = scalar_b - 2
    
    elif density > 100:
        scalar_a = scalar_a + 13
        scalar_b = scalar_b + 2
    else:
        print(f"Missed a case with density at = {density}")

    #run_yosys(verbose=False)
    
    with open("gds_stat.log", "a") as f:
        f.write(f"effective_density = {density}\n")
        f.write(f"wire_length = {wire_length}\n")
        f.write(f"dimensions = {dimensions}\n")
        f.write("\n")

    scalar_a, scalar_b = int(scalar_a), int(scalar_b)


It took yosys = 4.71s
9. Printing statistics.

=== rot ===

   Number of wires:                486
   Number of wire bits:           1751
   Number of public wires:          27
   Number of public wire bits:    1292
   Number of memories:               0
   Number of memory bits:            0
   Number of processes:              0
   Number of cells:                389
     inv_x0                          5
     mux2_x1                       384

Removed 0 unused cells and 149 unused wires.


Trying with a=325, b=50
Going to try with a=325, b=50
GDS: took = 6.55s . Effective desnsity =  12.0, Wire length = 50143
Going to try with a=41, b=6
GDS: took = 4.19s . Effective desnsity =  854.0, Wire length = None
Going to try with a=713, b=104
GDS: took = 14.58s . Effective desnsity =  2.62, Wire length = 87054
Going to try with a=531, b=77
GDS: took = 9.91s . Effective desnsity =  4.77, Wire length = 68877
Going to try with a=209, b=30
GDS: took = 4.63s . Effective desnsity =  30.9, Wire len

In [9]:

# def min_index_skip_None(L):
#     # first find an element of type int
#     for i in range(len(L)):
#         if isinstance(L[i], int):
#             idx = i
#             break # exit the loop

#     for i in range(len(L)):
#         try: 
#             if L[i] < L[idx]:
#                 idx = i
#         except:
#             pass # Skip none types

#     return idx


# min_index_skip_None([None, 1, 3, None, -1])

4